In [47]:
import pandas as pd
from sqlalchemy import create_engine, text 

def getTeamData(engine):
    with engine.connect() as conn:
        teamQuery = text("""
            WITH game_teams AS (
                SELECT DISTINCT
                    game_id,
                    season_id,
                    team_game_date as game_date,
                    MIN(team_id) as team1_id,
                    MAX(team_id) as team2_id
                FROM team_average_game_stats
                WHERE season_id IN ('22023', '22022')  -- Fixed season format
                GROUP BY game_id, season_id, team_game_date
            )
            SELECT 
                gt.game_id,
                gt.season_id,
                gt.game_date,
                gt.team1_id,
                gt.team2_id,
                
                -- Team 1 stats
                t1.avg_team_pts as team_pts,
                t1.avg_team_fgm as team_fgm,
                t1.avg_team_fga as team_fga,
                t1.avg_team_fg_pct as team_fg_pct,
                t1.avg_team_fg3m as team_fg3m,
                t1.avg_team_fg3a as team_fg3a,
                t1.avg_team_fg3_pct as team_fg3_pct,
                t1.avg_team_ftm as team_ftm,
                t1.avg_team_fta as team_fta,
                t1.avg_team_ft_pct as team_ft_pct,
                t1.avg_team_oreb as team_oreb,
                t1.avg_team_dreb as team_dreb,
                t1.avg_team_reb as team_reb,
                t1.avg_team_ast as team_ast,
                t1.avg_team_stl as team_stl,
                t1.avg_team_blk as team_blk,
                t1.avg_team_tov as team_tov,
                t1.avg_team_pf as team_pf,
                t1.games_in_average as team_games_in_avg,
                
                -- Opponent stats
                t2.avg_team_pts as opp_pts,
                t2.avg_team_fgm as opp_fgm,
                t2.avg_team_fga as opp_fga,
                t2.avg_team_fg_pct as opp_fg_pct,
                t2.avg_team_fg3m as opp_fg3m,
                t2.avg_team_fg3a as opp_fg3a,
                t2.avg_team_fg3_pct as opp_fg3_pct,
                t2.avg_team_ftm as opp_ftm,
                t2.avg_team_fta as opp_fta,
                t2.avg_team_ft_pct as opp_ft_pct,
                t2.avg_team_oreb as opp_oreb,
                t2.avg_team_dreb as opp_dreb,
                t2.avg_team_reb as opp_reb,
                t2.avg_team_ast as opp_ast,
                t2.avg_team_stl as opp_stl,
                t2.avg_team_blk as opp_blk,
                t2.avg_team_tov as opp_tov,
                t2.avg_team_pf as opp_pf,
                t2.games_in_average as opp_games_in_avg,
                
                -- Differences (team - opponent)
                t1.avg_team_pts - t2.avg_team_pts as diff_pts,
                t1.avg_team_fgm - t2.avg_team_fgm as diff_fgm,
                t1.avg_team_fga - t2.avg_team_fga as diff_fga,
                t1.avg_team_fg_pct - t2.avg_team_fg_pct as diff_fg_pct,
                t1.avg_team_fg3m - t2.avg_team_fg3m as diff_fg3m,
                t1.avg_team_fg3a - t2.avg_team_fg3a as diff_fg3a,
                t1.avg_team_fg3_pct - t2.avg_team_fg3_pct as diff_fg3_pct,
                t1.avg_team_ftm - t2.avg_team_ftm as diff_ftm,
                t1.avg_team_fta - t2.avg_team_fta as diff_fta,
                t1.avg_team_ft_pct - t2.avg_team_ft_pct as diff_ft_pct,
                t1.avg_team_oreb - t2.avg_team_oreb as diff_oreb,
                t1.avg_team_dreb - t2.avg_team_dreb as diff_dreb,
                t1.avg_team_reb - t2.avg_team_reb as diff_reb,
                t1.avg_team_ast - t2.avg_team_ast as diff_ast,
                t1.avg_team_stl - t2.avg_team_stl as diff_stl,
                t1.avg_team_blk - t2.avg_team_blk as diff_blk,
                t1.avg_team_tov - t2.avg_team_tov as diff_tov,
                t1.avg_team_pf - t2.avg_team_pf as diff_pf
                
            FROM game_teams gt
            JOIN team_average_game_stats t1 ON gt.game_id = t1.game_id AND gt.team1_id = t1.team_id
            JOIN team_average_game_stats t2 ON gt.game_id = t2.game_id AND gt.team2_id = t2.team_id
            WHERE t1.games_in_average >= 5
              AND t2.games_in_average >= 5
            ORDER BY gt.game_date, gt.game_id;
        """)
        return pd.read_sql(teamQuery, conn)        

def getPlayerData(engine):
    with engine.connect() as conn:
        playerQuery = text("""
            WITH game_teams AS (
                SELECT DISTINCT
                    game_id,
                    season_id,
                    MIN(team_id) as team1_id,
                    MAX(team_id) as team2_id
                FROM player_average_game_stats
                WHERE season_id IN ('22023', '22022')  -- Fixed season format
                GROUP BY game_id, season_id
            ),
            ranked_players AS (
                SELECT 
                    pgs.game_id,
                    pgs.season_id,
                    pgs.team_id,
                    pgs.player_id,
                    p.player_name,
                    pgs.avg_min,
                    pgs.avg_pts,
                    pgs.avg_fgm,
                    pgs.avg_fga,
                    pgs.avg_fg_pct,
                    pgs.avg_fg3m,
                    pgs.avg_fg3a,
                    pgs.avg_fg3_pct,
                    pgs.avg_ftm,
                    pgs.avg_fta,
                    pgs.avg_ft_pct,
                    pgs.avg_oreb,
                    pgs.avg_dreb,
                    pgs.avg_reb,
                    pgs.avg_ast,
                    pgs.avg_stl,
                    pgs.avg_blk,
                    pgs.avg_tov,
                    pgs.avg_pf,
                    pgs.avg_plus_minus,
                    pgs.games_in_average,
                    ROW_NUMBER() OVER (PARTITION BY pgs.game_id, pgs.team_id ORDER BY pgs.avg_min DESC) as player_rank
                FROM player_average_game_stats pgs
                JOIN players p ON pgs.player_id = p.player_id
                WHERE pgs.avg_min > 0
            )
            SELECT 
                gt.game_id,
                gt.season_id,
                'team' as player_team_type,
                gt.team1_id as team_id,
                rp.player_rank,
                rp.player_id,
                rp.player_name,
                rp.avg_min,
                rp.avg_pts,
                rp.avg_fgm,
                rp.avg_fga,
                rp.avg_fg_pct,
                rp.avg_fg3m,
                rp.avg_fg3a,
                rp.avg_fg3_pct,
                rp.avg_ftm,
                rp.avg_fta,
                rp.avg_ft_pct,
                rp.avg_oreb,
                rp.avg_dreb,
                rp.avg_reb,
                rp.avg_ast,
                rp.avg_stl,
                rp.avg_blk,
                rp.avg_tov,
                rp.avg_pf,
                rp.avg_plus_minus,
                rp.games_in_average
            FROM game_teams gt
            JOIN ranked_players rp ON gt.game_id = rp.game_id AND gt.team1_id = rp.team_id
            WHERE rp.player_rank <= 10
            
            UNION ALL
            
            SELECT 
                gt.game_id,
                gt.season_id,
                'opp' as player_team_type,
                gt.team2_id as team_id,
                rp.player_rank,
                rp.player_id,
                rp.player_name,
                rp.avg_min,
                rp.avg_pts,
                rp.avg_fgm,
                rp.avg_fga,
                rp.avg_fg_pct,
                rp.avg_fg3m,
                rp.avg_fg3a,
                rp.avg_fg3_pct,
                rp.avg_ftm,
                rp.avg_fta,
                rp.avg_ft_pct,
                rp.avg_oreb,
                rp.avg_dreb,
                rp.avg_reb,
                rp.avg_ast,
                rp.avg_stl,
                rp.avg_blk,
                rp.avg_tov,
                rp.avg_pf,
                rp.avg_plus_minus,
                rp.games_in_average
            FROM game_teams gt
            JOIN ranked_players rp ON gt.game_id = rp.game_id AND gt.team2_id = rp.team_id
            WHERE rp.player_rank <= 10
            
            ORDER BY game_id, player_team_type, player_rank;
        """)
        return pd.read_sql(playerQuery, conn)

In [50]:
def remove_leakage(df, drop_cols):
    cols_to_drop = [c for c in df.columns if c in drop_cols]
    return df.drop(columns=cols_to_drop, errors='ignore')


In [52]:
from sklearn.feature_selection import VarianceThreshold

def variance_filter(df, target='target_win', threshold=0.001):
    numeric_df = df.drop(columns=[target]).select_dtypes(include=['int64','float64'])
    
    selector = VarianceThreshold(threshold=threshold)
    selector.fit(numeric_df)

    low_variance = numeric_df.columns[~selector.get_support()]
    print("🔎 Low-variance removed:", low_variance.tolist())

    return df.drop(columns=low_variance)

In [54]:
import numpy as np

def correlation_pruning(df, target='target_win', threshold=0.90):
    numeric_df = df.drop(columns=[target]).select_dtypes(include=['int64','float64'])

    corr = numeric_df.corr().abs()

    # Upper triangle mask
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    to_drop = [
        column for column in upper.columns
        if any(upper[column] > threshold)
    ]

    print("🔎 High-correlation removed:", to_drop)
    return df.drop(columns=to_drop)


In [56]:
import pandas as pd
from sklearn.feature_selection import f_classif

def anova_scores(df, target='target_win'):
    X = df.drop(columns=[target])
    y = df[target]

    f_scores, p_vals = f_classif(X, y)

    anova_df = (
        pd.DataFrame({
            'feature': X.columns,
            'f_score': f_scores
        })
        .sort_values('f_score', ascending=False)
        .reset_index(drop=True)
    )

    print("🏅 Top ANOVA features:")
    print(anova_df.head(20))
    return anova_df


In [58]:
from sklearn.feature_selection import mutual_info_classif

def mi_scores(df, target='target_win'):
    X = df.drop(columns=[target])
    y = df[target]

    mi = mutual_info_classif(X, y)

    mi_df = (
        pd.DataFrame({
            'feature': X.columns,
            'mi_score': mi
        })
        .sort_values('mi_score', ascending=False)
        .reset_index(drop=True)
    )

    print("🏅 Top MI features:")
    print(mi_df.head(20))
    return mi_df


In [60]:
def flatten_player_data(player_df):
    """
    Pivot player data from long format (multiple rows per game) 
    to wide format (one row per game with player stats as columns).
    
    Returns: DataFrame with one row per game_id
    """
    # Define which columns are stats (exclude identifiers)
    stat_cols = [
        'avg_min', 'avg_pts', 'avg_fgm', 'avg_fga', 'avg_fg_pct',
        'avg_fg3m', 'avg_fg3a', 'avg_fg3_pct', 'avg_ftm', 'avg_fta', 
        'avg_ft_pct', 'avg_oreb', 'avg_dreb', 'avg_reb', 'avg_ast',
        'avg_stl', 'avg_blk', 'avg_tov', 'avg_pf', 'avg_plus_minus',
        'games_in_average'
    ]
    
    # Create a column name for each player stat
    # Format: {team_type}_player{rank}_{stat}
    player_df['player_prefix'] = (
        player_df['player_team_type'] + '_player' + 
        player_df['player_rank'].astype(str)
    )
    
    # Pivot: each (game_id, player_prefix, stat) becomes a column
    pivoted_parts = []
    
    for stat in stat_cols:
        stat_pivot = player_df.pivot_table(
            index='game_id',
            columns='player_prefix',
            values=stat,
            aggfunc='first'
        )
        # Rename columns to include stat name
        stat_pivot.columns = [f"{col}_{stat}" for col in stat_pivot.columns]
        pivoted_parts.append(stat_pivot)
    
    # Concatenate all stat pivots horizontally
    flattened = pd.concat(pivoted_parts, axis=1)
    
    # Reset index to make game_id a column again
    flattened = flattened.reset_index()
    
    return flattened


def create_ml_dataset(engine):
    """
    Create the full ML dataset by combining team and player features.
    Returns one row per game with all features flattened.
    """
    # Get team data
    team_df = getTeamData(engine)
    
    # Get player data
    player_df = getPlayerData(engine)
    
    # Flatten player data
    player_flat = flatten_player_data(player_df)
    
    # Merge team and player data on game_id
    ml_df = team_df.merge(player_flat, on='game_id', how='inner')
    
    print(f"✅ Dataset shape: {ml_df.shape}")
    print(f"   - {len(ml_df)} games")
    print(f"   - {len(ml_df.columns)} total features")
    
    return ml_df

In [62]:
def add_target_variable(ml_df, engine):
    """
    Add the target variable (whether team1 won) to the dataset.
    """
    with engine.connect() as conn:
        # Get actual game results for all teams
        results_query = text("""
            SELECT 
                game_id,
                team_id,
                team_pts
            FROM team_game_stats
            WHERE season_id IN ('22023', '22022')
        """)
        results_df = pd.read_sql(results_query, conn)
    
    # Join to get both teams' actual scores
    team1_results = results_df.rename(columns={'team_id': 'team1_id', 'team_pts': 'team1_actual_pts'})
    team2_results = results_df.rename(columns={'team_id': 'team2_id', 'team_pts': 'team2_actual_pts'})
    
    ml_df = ml_df.merge(team1_results[['game_id', 'team1_id', 'team1_actual_pts']], 
                        on=['game_id', 'team1_id'], how='left')
    ml_df = ml_df.merge(team2_results[['game_id', 'team2_id', 'team2_actual_pts']], 
                        on=['game_id', 'team2_id'], how='left')
    
    # Create target: 1 if team1 won, 0 if team2 won
    ml_df['target_win'] = (ml_df['team1_actual_pts'] > ml_df['team2_actual_pts']).astype(int)
    
    # Drop the actual score columns (they're leakage)
    ml_df = ml_df.drop(columns=['team1_actual_pts', 'team2_actual_pts'])
    
    return ml_df

In [64]:
LEAKAGE_COLS = [
    'game_id', 'season_id', 'game_date', 'team1_id', 'team2_id',
    'team_games_in_avg', 'opp_games_in_avg'
]

def preprocess_features_detailed(df):
    """
    Full feature selection pipeline with intermediate results.
    """
    print("\n🚀 Starting feature selection pipeline...\n")
    
    # Store original
    original_df = df.copy()
    
    # 1. Remove leakage columns
    df = df.drop(columns=[col for col in LEAKAGE_COLS if col in df.columns])
    print(f"✓ Removed {len([col for col in LEAKAGE_COLS if col in original_df.columns])} leakage columns")
    
    # 2. Drop non-numeric leftover columns
    df = df.select_dtypes(include=['int64','float64','float32']).copy()
    print(f"✓ Kept only numeric columns")
    
    # 3. Variance filtering
    after_variance = variance_filter(df)
    
    # 4. Correlation pruning
    after_correlation = correlation_pruning(after_variance)
    
    # 5. Relevance rankings
    anova_df = anova_scores(after_correlation)
    mi_df = mi_scores(after_correlation)
    
    # Return everything for analysis
    return {
        'original': original_df,
        'after_variance': after_variance,
        'after_correlation': after_correlation,
        'final': after_correlation,
        'anova': anova_df,
        'mi': mi_df
    }


In [66]:
#analyzing functions:
# Add these cells to your notebook

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def analyze_feature_selection_results(original_df, filtered_df, anova_df, mi_df):
    """
    Comprehensive analysis of feature selection results.
    Shows what was dropped, correlations, and feature importance.
    """
    print("=" * 80)
    print("FEATURE SELECTION SUMMARY")
    print("=" * 80)
    
    # 1. Overall reduction
    original_features = [col for col in original_df.columns if col != 'target_win']
    final_features = [col for col in filtered_df.columns if col != 'target_win']
    
    print(f"\n📊 FEATURE REDUCTION:")
    print(f"   Original features: {len(original_features)}")
    print(f"   Final features: {len(final_features)}")
    print(f"   Reduction: {len(original_features) - len(final_features)} features ({(1 - len(final_features)/len(original_features))*100:.1f}%)")
    
    # 2. Top features by ANOVA
    print(f"\n🏆 TOP 20 FEATURES (ANOVA F-Score):")
    print(anova_df.head(20).to_string(index=False))
    
    # 3. Top features by Mutual Information
    print(f"\n🏆 TOP 20 FEATURES (Mutual Information):")
    print(mi_df.head(20).to_string(index=False))
    
    # 4. Feature categories breakdown
    print(f"\n📁 FEATURE CATEGORIES REMAINING:")
    analyze_feature_categories(final_features)
    
    return final_features


def analyze_feature_categories(features):
    """
    Break down features by category (team, opp, diff, player stats).
    """
    categories = {
        'team_stats': [f for f in features if f.startswith('team_') and not f.startswith('team_player')],
        'opp_stats': [f for f in features if f.startswith('opp_') and not f.startswith('opp_player')],
        'diff_stats': [f for f in features if f.startswith('diff_')],
        'team_player_stats': [f for f in features if f.startswith('team_player')],
        'opp_player_stats': [f for f in features if f.startswith('opp_player')],
    }
    
    for cat, feats in categories.items():
        print(f"   {cat:20s}: {len(feats):4d} features")


def plot_correlation_heatmap(df, top_n=30, figsize=(16, 14)):
    """
    Plot correlation heatmap for top N features by ANOVA score.
    """
    # Get top features
    X = df.drop(columns=['target_win'])
    y = df['target_win']
    
    from sklearn.feature_selection import f_classif
    f_scores, _ = f_classif(X, y)
    
    top_indices = np.argsort(f_scores)[-top_n:]
    top_features = X.columns[top_indices].tolist()
    
    # Calculate correlation matrix
    corr = df[top_features + ['target_win']].corr()
    
    # Plot
    plt.figure(figsize=figsize)
    sns.heatmap(corr, 
                cmap='RdBu_r', 
                center=0, 
                vmin=-1, vmax=1,
                square=True,
                linewidths=0.5,
                cbar_kws={"shrink": 0.8})
    plt.title(f'Correlation Heatmap: Top {top_n} Features + Target', fontsize=14, pad=20)
    plt.tight_layout()
    plt.show()
    
    print(f"\n🔥 HIGHEST CORRELATIONS WITH TARGET:")
    target_corr = corr['target_win'].drop('target_win').abs().sort_values(ascending=False)
    print(target_corr.head(15))


def analyze_dropped_features(original_df, after_variance_df, after_correlation_df):
    """
    Detailed analysis of what was dropped at each step.
    """
    print("\n" + "=" * 80)
    print("DETAILED DROPOUT ANALYSIS")
    print("=" * 80)
    
    original_features = set(original_df.columns) - {'target_win'}
    after_variance = set(after_variance_df.columns) - {'target_win'}
    after_correlation = set(after_correlation_df.columns) - {'target_win'}
    
    # Variance filter dropouts
    variance_dropped = original_features - after_variance
    print(f"\n❌ DROPPED BY VARIANCE FILTER ({len(variance_dropped)} features):")
    if variance_dropped:
        for feat in sorted(variance_dropped):
            var = original_df[feat].var()
            print(f"   {feat:50s} (variance: {var:.6f})")
    else:
        print("   None")
    
    # Correlation filter dropouts
    correlation_dropped = after_variance - after_correlation
    print(f"\n❌ DROPPED BY CORRELATION FILTER ({len(correlation_dropped)} features):")
    if correlation_dropped:
        # Show what each was correlated with
        for feat in sorted(correlation_dropped):
            print(f"   {feat}")
    else:
        print("   None")


def plot_feature_importance_comparison(anova_df, mi_df, top_n=20):
    """
    Compare ANOVA and MI rankings side-by-side.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # ANOVA scores
    top_anova = anova_df.head(top_n)
    axes[0].barh(range(len(top_anova)), top_anova['f_score'])
    axes[0].set_yticks(range(len(top_anova)))
    axes[0].set_yticklabels(top_anova['feature'], fontsize=8)
    axes[0].invert_yaxis()
    axes[0].set_xlabel('F-Score')
    axes[0].set_title(f'Top {top_n} Features: ANOVA F-Score')
    axes[0].grid(axis='x', alpha=0.3)
    
    # MI scores
    top_mi = mi_df.head(top_n)
    axes[1].barh(range(len(top_mi)), top_mi['mi_score'])
    axes[1].set_yticks(range(len(top_mi)))
    axes[1].set_yticklabels(top_mi['feature'], fontsize=8)
    axes[1].invert_yaxis()
    axes[1].set_xlabel('MI Score')
    axes[1].set_title(f'Top {top_n} Features: Mutual Information')
    axes[1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()


def analyze_player_vs_team_importance(anova_df, mi_df):
    """
    Compare importance of player stats vs team stats.
    """
    print("\n" + "=" * 80)
    print("PLAYER VS TEAM FEATURE IMPORTANCE")
    print("=" * 80)
    
    def categorize_and_score(df, score_col):
        df = df.copy()
        df['category'] = df['feature'].apply(lambda x: 
            'team_player' if 'team_player' in x else
            'opp_player' if 'opp_player' in x else
            'team_stats' if x.startswith('team_') else
            'opp_stats' if x.startswith('opp_') else
            'diff_stats' if x.startswith('diff_') else 'other'
        )
        return df.groupby('category')[score_col].agg(['mean', 'sum', 'count'])
    
    print("\n📊 BY ANOVA F-SCORE:")
    anova_summary = categorize_and_score(anova_df, 'f_score')
    print(anova_summary.sort_values('mean', ascending=False))
    
    print("\n📊 BY MUTUAL INFORMATION:")
    mi_summary = categorize_and_score(mi_df, 'mi_score')
    print(mi_summary.sort_values('mean', ascending=False))


# Modified preprocess_features to return intermediate results
def preprocess_features_detailed(df):
    """
    Full feature selection pipeline with intermediate results.
    """
    print("\n🚀 Starting feature selection pipeline...\n")
    
    # Store original
    original_df = df.copy()
    
    # 1. Remove leakage columns
    df = df.drop(columns=[col for col in LEAKAGE_COLS if col in df.columns])
    print(f"✓ Removed {len([col for col in LEAKAGE_COLS if col in original_df.columns])} leakage columns")
    
    # 2. Drop non-numeric leftover columns
    df = df.select_dtypes(include=['int64','float64','float32']).copy()
    print(f"✓ Kept only numeric columns")
    
    # 3. Variance filtering
    after_variance = variance_filter(df)
    
    # 4. Correlation pruning
    after_correlation = correlation_pruning(after_variance)
    
    # 5. Relevance rankings
    anova_df = anova_scores(after_correlation)
    mi_df = mi_scores(after_correlation)
    
    # Return everything for analysis
    return {
        'original': original_df,
        'after_variance': after_variance,
        'after_correlation': after_correlation,
        'final': after_correlation,
        'anova': anova_df,
        'mi': mi_df
    }

In [43]:
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import pandas as pd
import os

load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

ml_df = create_ml_dataset(engine)

NameError: name 'create_ml_dataset' is not defined

In [ ]:
# 2. Add target variable
ml_df = add_target_variable(ml_df, engine)

In [ ]:
# 3. Run detailed feature selection with comprehensive analysis
results = preprocess_features_detailed(ml_df)

# Comprehensive analysis
final_features = analyze_feature_selection_results(
    results['original'], 
    results['final'], 
    results['anova'], 
    results['mi']
)

# Detailed dropout analysis
analyze_dropped_features(
    results['original'],
    results['after_variance'],
    results['after_correlation']
)

# Player vs Team comparison
analyze_player_vs_team_importance(results['anova'], results['mi'])

# Visualizations
plot_feature_importance_comparison(results['anova'], results['mi'], top_n=25)
plot_correlation_heatmap(results['final'], top_n=30)

# Store final filtered dataframe
filtered_df = results['final']